# Занятие 9. Мини-проект с базой данных sqlite3

## Цель занятия

На прошлых занятиях мы изучили:

- создание базы данных;
- создание таблиц;
- `INSERT`;
- `SELECT`;
- `UPDATE`;
- `DELETE`;
- `WHERE`;
- `ORDER BY`;
- `JOIN`;
- `GROUP BY`;
- функции Python для работы с sqlite3.

Сегодня собираем всё вместе в **мини-проект с базой данных**.

---

## Главная идея занятия

До этого мы изучали отдельные SQL-команды.

Теперь задача другая:

> собрать маленькое приложение, где Python-функции работают с базой данных как единый проект.

---

## Какой мини-проект делаем в solved

В solved-ноутбуке делаем мини-проект:

**HR Resume Tracker**

Он умеет:

- создать таблицу кандидатов;
- добавить кандидатов;
- показать всех кандидатов;
- найти кандидатов по статусу;
- обновить статус кандидата;
- посчитать аналитику по статусам;
- показать лучших кандидатов по score.

---

## Почему это полезно в реальной жизни

Похожая логика есть в реальных HR-системах:

- кандидат откликнулся;
- рекрутер смотрит список кандидатов;
- меняет статус;
- выбирает лучших;
- считает аналитику по воронке отбора.

---

## Как это связано с дипломом

Это почти готовая логика дипломного проекта.

По такой же схеме можно сделать:

HR:
- кандидаты;
- вакансии;
- статусы;
- score.

Питание:
- продукты;
- приёмы пищи;
- калории;
- рекомендации.

Спорт:
- тренировки;
- упражнения;
- длительность;
- прогресс.

Магазин:
- товары;
- заказы;
- клиенты;
- выручка.

---

## Что должно получиться

После занятия студент должен понимать:

- как собрать SQL-код в функции;
- как сделать главный сценарий проекта;
- как показать результат преподавателю;
- как подготовить проект к экзамену;
- что переносить из Colab в VS Code.

---

## Связь с GitHub

На этом занятии студент должен сохранить:

- solved-ноутбук;
- TODO-ноутбук;
- свою базу данных;
- функции мини-проекта;
- будущий код для `src/db/`.

Рекомендуемая ветка:

```bash
git checkout main
git pull origin main
git checkout -b feature/lesson-09-db-mini-project
```

Рекомендуемый commit:

```bash
git add .
git commit -m "feat: build sqlite mini project"
git push -u origin feature/lesson-09-db-mini-project
```

## Ячейка 1. Подключаем sqlite3 и создаём подключение

Начинаем мини-проект с функции подключения к базе данных.

Это хорошая проектная привычка: не писать `sqlite3.connect(...)` везде вручную, а вынести подключение в функцию.

In [ ]:
import sqlite3

def get_connection(db_name="hr_resume_tracker.db"):
    connection = sqlite3.connect(db_name)
    return connection

connection = get_connection()
cursor = connection.cursor()

print("База данных подключена")
assert connection is not None

## Ячейка 2. Создаём таблицу candidates

Таблица `candidates` хранит кандидатов.

Поля:

- `id` — уникальный номер;
- `full_name` — ФИО кандидата;
- `position` — позиция;
- `city` — город;
- `experience_years` — опыт;
- `status` — статус;
- `score` — оценка кандидата.

In [ ]:
def create_candidates_table(connection):
    cursor = connection.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS candidates (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        full_name TEXT,
        position TEXT,
        city TEXT,
        experience_years INTEGER,
        status TEXT,
        score REAL
    )
    """)
    connection.commit()

create_candidates_table(connection)

print("Таблица candidates создана")

## Ячейка 3. Очищаем таблицу

Чтобы при повторном запуске ноутбука данные не дублировались, очищаем таблицу.

In [ ]:
def clear_candidates(connection):
    cursor = connection.cursor()
    cursor.execute("DELETE FROM candidates")
    connection.commit()

clear_candidates(connection)

print("Таблица candidates очищена")

## Ячейка 4. Добавляем кандидатов

Создаём функцию `add_candidate`.

Она добавляет одну запись в базу данных.

In [ ]:
def add_candidate(connection, full_name, position, city, experience_years, status, score):
    cursor = connection.cursor()
    cursor.execute(
        """
        INSERT INTO candidates (full_name, position, city, experience_years, status, score)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (full_name, position, city, experience_years, status, score)
    )
    connection.commit()

add_candidate(connection, "Анна Смирнова", "Python Developer", "Казань", 2, "new", 87.5)
add_candidate(connection, "Иван Петров", "Data Analyst", "Москва", 1, "interview", 78.0)
add_candidate(connection, "Ольга Иванова", "Python Developer", "Казань", 3, "offer", 92.0)
add_candidate(connection, "Павел Соколов", "QA Engineer", "Уфа", 1, "new", 70.0)
add_candidate(connection, "Мария Кузнецова", "ML Intern", "Москва", 0, "screening", 81.0)

print("Кандидаты добавлены")

## Ячейка 5. Получаем всех кандидатов

Функция `get_all_candidates` возвращает все записи.

In [ ]:
def get_all_candidates(connection):
    cursor = connection.cursor()
    cursor.execute("SELECT * FROM candidates")
    return cursor.fetchall()

candidates = get_all_candidates(connection)

for candidate in candidates:
    print(candidate)

assert len(candidates) == 5

## Ячейка 6. Поиск кандидатов по статусу

Функция `find_candidates_by_status` ищет кандидатов по статусу.

Это похоже на фильтр в реальной HR-системе.

In [ ]:
def find_candidates_by_status(connection, status):
    cursor = connection.cursor()
    cursor.execute(
        "SELECT * FROM candidates WHERE status = ?",
        (status,)
    )
    return cursor.fetchall()

new_candidates = find_candidates_by_status(connection, "new")

print("Новые кандидаты:")
for candidate in new_candidates:
    print(candidate)

assert len(new_candidates) == 2

## Ячейка 7. Обновляем статус кандидата

Функция `update_candidate_status` меняет статус кандидата по имени.

Важно: используем `WHERE`, чтобы изменить только одного кандидата.

In [ ]:
def update_candidate_status(connection, full_name, new_status):
    cursor = connection.cursor()
    cursor.execute(
        "UPDATE candidates SET status = ? WHERE full_name = ?",
        (new_status, full_name)
    )
    connection.commit()

update_candidate_status(connection, "Павел Соколов", "interview")

updated = find_candidates_by_status(connection, "interview")

print("Кандидаты на интервью:")
for candidate in updated:
    print(candidate)

assert len(updated) >= 2

## Ячейка 8. Аналитика по статусам

Считаем, сколько кандидатов находится в каждом статусе.

Это уже маленький аналитический отчёт.

In [ ]:
def get_status_report(connection):
    cursor = connection.cursor()
    cursor.execute("""
    SELECT status, COUNT(*)
    FROM candidates
    GROUP BY status
    ORDER BY COUNT(*) DESC
    """)
    return cursor.fetchall()

status_report = get_status_report(connection)

print("Отчёт по статусам:")
for row in status_report:
    print(row)

assert len(status_report) >= 1

## Ячейка 9. Топ кандидатов по score

Получим лучших кандидатов по оценке.

Это полезно для первичного отбора.

In [ ]:
def get_top_candidates(connection, limit=3):
    cursor = connection.cursor()
    cursor.execute(
        "SELECT full_name, position, city, score FROM candidates ORDER BY score DESC LIMIT ?",
        (limit,)
    )
    return cursor.fetchall()

top_candidates = get_top_candidates(connection, 3)

print("Топ кандидатов:")
for candidate in top_candidates:
    print(candidate)

assert len(top_candidates) == 3
assert top_candidates[0][3] >= top_candidates[-1][3]

## Ячейка 10. Финальный сценарий проекта

Собираем итоговый вывод мини-проекта.

Затем закрываем соединение.

In [ ]:
def show_project_summary(connection):
    print("=== HR Resume Tracker ===")
    print()

    print("Все кандидаты:")
    for candidate in get_all_candidates(connection):
        print(candidate)

    print("\nОтчёт по статусам:")
    for row in get_status_report(connection):
        print(row)

    print("\nТоп кандидатов:")
    for candidate in get_top_candidates(connection, 3):
        print(candidate)

show_project_summary(connection)

connection.close()

print("\nСоединение закрыто")

# Итог занятия 9

Сегодня вы собрали мини-проект с базой данных.

Вы научились:

- создавать таблицу проекта;
- писать функции для SQL;
- добавлять данные;
- фильтровать данные;
- обновлять данные;
- строить аналитический отчёт;
- собирать главный сценарий проекта.

## Что переносить в VS Code

```text
src/db/connection.py
src/db/create_tables.py
src/db/insert_data.py
src/db/queries.py
src/db/reports.py
src/main.py
```

## Что коммитить в GitHub

```bash
git add .
git commit -m "feat: build sqlite mini project"
git push -u origin feature/lesson-09-db-mini-project
```